<a href="https://colab.research.google.com/github/bintangF05/modul4/blob/main/modul4ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

lab 4.1

In [1]:
!pip install unsloth
!pip install trl datasets
!pip install accelerate --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 767.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6

In [2]:
# ==============================================================================
# 0. SETTING ENVIRONMENT VARIABLES (Wajib ditaruh di paling atas sebelum import)
# ==============================================================================
import os
os.environ["TORCH_COMPILE_BACKEND"] = "eager"
os.environ["TORCH_DYNAMO_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # Optimasi alokasi VRAM

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ==============================================================================
# 1. CONFIGURATION & MODEL LOADING (QLoRA 4-bit)
# ==============================================================================
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048 # Sesuai dengan panjang sekuens SFTTrainer Anda
LORA_RANK = 16    # Efisiensi rank LoRA
LORA_ALPHA = 16
OUTPUT_DIR = "./lora_finetuned_model"

# Memuat model dasar dengan Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,       # Otomatis mendeteksi tipe data komponen hardware
    load_in_4bit = True # Mengaktifkan QLoRA 4-bit untuk menghemat VRAM
)

# Menerapkan PEFT LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], # Target matriks attention
    lora_dropout = 0.05,
    use_gradient_checkpointing = True,
)

# ==============================================================================
# 2. DATASET PREPARATION (Bebas Error Pickling)
# ==============================================================================
# Memuat 100 sampel data dari OpenHermes untuk simulasi cepat
dataset = load_dataset("teknium/OpenHermes-2.5", split="train[:100]")

# Fungsi bernama untuk mengekstrak percakapan DAN mengubah ke format ChatML sekaligus
def format_dataset_aman(example):
    conversations = example["conversations"]
    question = conversations[0]["value"]
    answer = conversations[1]["value"]

    # Mengembalikan teks struktur standar instruksi ChatML
    return {
        "text": f"<|user|>\n{question}<|end|>\n<|assistant|>\n{answer}<|end|>"
    }

# Jalankan map secara aman tanpa fungsi lambda anonim
dataset = dataset.map(format_dataset_aman, remove_columns=dataset.column_names)

# ==============================================================================
# 3. TRAINING ARGUMENTS & SFT SETUP
# ==============================================================================
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 1,     # Diperkecil untuk mengamankan VRAM laptop/Colab
    gradient_accumulation_steps = 8,     # Ditingkatkan agar batch size efektif tetap terjaga
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = True,                         # Memastikan training berjalan di presisi 16-bit
    logging_steps = 10,
    save_strategy = "no",                # Mematikan autosave untuk mencegah PicklingError
    save_total_limit = 0,                # Memastikan tidak ada checkpoint per epoch yang disimpan
    optim = "adamw_8bit",                # Penggunaan optimizer hemat memori
    torch_compile = False,               # Menolak kompilasi Dynamo otomatis untuk mencegah crash node FX
    push_to_hub = False,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
    max_seq_length = 2048,               # Menyelaraskan max_seq_length dengan model konfigurasi
    dataset_text_field = "text",
)

# ==============================================================================
# 4. EXECUTE TRAINING & INFERENCE TEST
# ==============================================================================
print("Memulai proses fine-tuning (Lab 4.1)...")
trainer.train()

# Menyimpan adapter LoRA yang sudah dilatih
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters berhasil disimpan di folder: {OUTPUT_DIR}")

# Mengubah model ke mode inferensi cepat
model = FastLanguageModel.for_inference(model)

# Menguji model dengan prompt baru
prompt = "<|user|>\nWhat is the capital of France?<|end|>\n<|assistant|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.7)

print("\n=== Generated Response ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

KeyboardInterrupt: 

lab 4.2

In [ ]:
!pip install diffusers transformers accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

# Konfigurasi Model dan Hardware
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256  # Sesuai modul, menggunakan ukuran 256 agar cepat di Colab

print("Sedang memuat pipeline Stable Diffusion ke GPU T4...")
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype = torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker = None  # Menonaktifkan safety checker untuk menghemat waktu inferensi
)
pipe = pipe.to(DEVICE)
print("Pipeline berhasil dimuat!")

In [ ]:
# Daftar prompt teks sesuai instruksi modul
prompts = [
    "A serene mountain landscape at sunset, photorealistic, 4k",
    "A futuristic city with flying cars, neon lights, cyberpunk style",
    "A cat wearing a spacesuit floating in space, digital art"
]

for i, prompt in enumerate(prompts):
    print(f"\n[Gambar {i+1}/{len(prompts)}] Sedang merender: {prompt}")
    image = pipe(
        prompt = prompt,
        negative_prompt = "blurry, low quality, distorted", # Menghilangkan elemen buram
        height = IMG_SIZE,
        width = IMG_SIZE,
        num_inference_steps = 30,  # 30 langkah denoising (ideal untuk kualitas vs kecepatan)
        guidance_scale = 7.5,      # Nilai standar kepatuhan model terhadap teks prompt
        num_images_per_prompt = 1,
        generator = torch.manual_seed(42), # Mengunci seed agar hasilnya konsisten
    ).images[0]

    # Menyimpan otomatis ke penyimpanan lokal cloud Colab
    filename = f"output_{i+1}.png"
    image.save(filename)
    print(f"Berhasil menyimpan file gambar: {filename}")

In [ ]:
test_prompt = "A dragon breathing fire, fantasy art"
guidance_scales = [2.0, 7.5, 15.0] # Variasi skala yang diminta oleh modul

print(f"Memulai eksperimen parameter Guidance Scale untuk prompt: '{test_prompt}'")

for gs in guidance_scales:
    image = pipe(
        prompt = test_prompt,
        height = IMG_SIZE,
        width = IMG_SIZE,
        num_inference_steps = 25,
        guidance_scale = gs,
        generator = torch.manual_seed(42), # Mengunci seed yang sama agar perbandingannya adil
    ).images[0]

    filename = f"dragon_gs_{gs}.png"
    image.save(filename)
    print(f"Gambar naga dengan skala {gs} tersimpan di: {filename}")

print("\nSeluruh tahapan Lab 4.2 selesai!")

In [ ]:
from google.colab import files

# Mengunduh gambar hasil utama
files.download('output_1.png')
files.download('output_2.png')
files.download('output_3.png')

# Mengunduh gambar hasil eksperimen bahan analisis
files.download('dragon_gs_2.0.png')
files.download('dragon_gs_7.5.png')
files.download('dragon_gs_15.0.png')

lab 4.3

In [ ]:
from google.colab import drive
import os
import zipfile

print("=== [MULAI] Menghubungkan Google Drive & Ekstraksi Ramah RAM ===")

# 1. Menghubungkan Google Colab ke Google Drive kamu
drive.mount('/content/drive')

# 2. PATH DISESUAIKAN: Menuju folder 'mydrive' sesuai gambar kamu
zip_path = "/content/drive/MyDrive/mydrive/photos.zip"
extract_path = "./data/photos/"

# Memastikan file benar-benar ada di Drive sebelum diproses
if not os.path.exists(zip_path):
    print(f"\n[ERROR] File TIDAK ditemukan di: {zip_path}")
    print("Silakan cek kembali apakah nama folder 'mydrive' di Drive Anda sudah ditulis dengan huruf kecil/besar yang benar.")
    raise FileNotFoundError("photos.zip missing di folder mydrive")

print(f"\nFile ditemukan! Ukuran file cukup besar (4GB+).")
print(f"Memulai ekstraksi aman secara bertahap ke {extract_path}...")
os.makedirs(extract_path, exist_ok=True)

# 3. OPTIMASI RAM: Mengekstrak file satu per satu (Stream Extraction)
# Cara ini tidak akan memakan RAM Colab karena file dicicil, bukan langsung dimuat sekaligus ke memori.
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    files_in_zip = zip_ref.namelist()
    total_files = len(files_in_zip)

    print(f"Total terdapat {total_files} file di dalam ZIP.")
    for index, file_name in enumerate(files_in_zip):
        zip_ref.extract(file_name, extract_path)

        # Menampilkan progress setiap 50 file agar Anda tahu proses tetap berjalan
        if index % 50 == 0 or index == total_files - 1:
            print(f"Progress Ekstraksi: [{index + 1}/{total_files}] file selesai...")

print("\n=== [SUKSES] Proses unzip selesai dengan aman tanpa crash RAM! ===")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

lab 4.4

In [ ]:
# ==============================================================================
# LAB 4.4: TINY DDPM ON 2D TOY DATA (CPU-FRIENDLY)
# ==============================================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# ------------------------------------------------------------------------------
# 1. CONFIGURATION SETUP
# ------------------------------------------------------------------------------
T = 1000               # Jumlah total tahapan difusi (diffusion steps)
BETA_START = 1e-4      # Nilai beta awal schedule linear
BETA_END = 0.02        # Nilai beta akhir schedule linear
DEVICE = "cpu"         # Dirancang ramah untuk running di CPU lokal
HIDDEN_DIM = 128
EPOCHS = 5000          # Jumlah epoch pelatihan
BATCH_SIZE = 256

# ------------------------------------------------------------------------------
# 2. GENERATE TOY DATA (Pola Dua Bulan Sabit)
# ------------------------------------------------------------------------------
def generate_data(n_samples=10000):
    """Membangkitkan data mainan 2D berbentuk pola two moons."""
    X, _ = make_moons(n_samples=n_samples, noise=0.05)
    return torch.tensor(X, dtype=torch.float32)

data = generate_data(10000).to(DEVICE)
print(f"Data asli berhasil dibuat dengan bentuk tensor: {data.shape}")

# ------------------------------------------------------------------------------
# 3. DEFINE DIFFUSION SCHEDULE (Matematika Forward Process)
# ------------------------------------------------------------------------------
betas = torch.linspace(BETA_START, BETA_END, T, device=DEVICE)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# Menghitung nilai akar kuadrat kumulatif untuk mempermudah kalkulasi noise
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# ------------------------------------------------------------------------------
# 4. FORWARD DIFFUSION FUNCTION (Proses Menambah Noise)
# ------------------------------------------------------------------------------
def q_sample(x0, t, noise=None):
    """Menambahkan noise secara acak sesuai dengan tahapan waktu t."""
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alpha_t = sqrt_alphas_cumprod[t].view(-1, 1)
    sqrt_one_minus_alpha_t = sqrt_one_minus_alphas_cumprod[t].view(-1, 1)

    return sqrt_alpha_t * x0 + sqrt_one_minus_alpha_t * noise, noise

# ------------------------------------------------------------------------------
# 5. DENOISING MODEL ARCHITECTURE (Simple MLP)
# ------------------------------------------------------------------------------
class NoisePredictionMLP(nn.Module):
    """Jaringan saraf tiruan sederhana untuk memprediksi besaran noise."""
    def __init__(self, input_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        # Embedding untuk tahapan waktu t
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim),
        )
        # Lapisan fully-connected utama
        self.net = nn.Sequential(
            nn.Linear(input_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x, t):
        t_emb = self.time_embed(t.float().unsqueeze(-1) / T)
        x_cat = torch.cat([x, t_emb], dim=-1)
        return self.net(x_cat)

model = NoisePredictionMLP(hidden_dim=HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ------------------------------------------------------------------------------
# 6. TRAINING LOOP
# ------------------------------------------------------------------------------
print("\nMemulai proses training model difusi (Lab 4.4)...")
for epoch in range(EPOCHS):
    # Mengambil sampel acak dari dataset utama
    idx = torch.randint(0, len(data), (BATCH_SIZE,))
    x0 = data[idx]

    # Mengambil tahapan waktu acak dari 0 sampai T
    t = torch.randint(0, T, (BATCH_SIZE,), device=DEVICE)

    # Forward diffusion: Dapatkan titik data terkontaminasi noise
    xt, noise = q_sample(x0, t)

    # Prediksi noise yang ada di dalam gambar menggunakan MLP
    noise_pred = model(xt, t)

    # Menghitung nilai Mean Squared Error (MSE) Loss
    loss = nn.functional.mse_loss(noise_pred, noise)

    # Proses Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d}/{EPOCHS} | Train Loss: {loss.item():.6f}")

print("Proses training selesai!")

# ------------------------------------------------------------------------------
# 7. SAMPLING FUNCTION (Reverse Process dari Pure Noise ke Pola Asli)
# ------------------------------------------------------------------------------
@torch.no_grad()
def p_sample(model, x, t):
    """Menghitung pengurangan noise secara berangsur-angsur di satu langkah waktu."""
    t_tensor = torch.full((x.shape[0],), t, device=DEVICE, dtype=torch.long)
    noise_pred = model(x, t_tensor)

    alpha_t = alphas[t]
    alpha_bar_t = alphas_cumprod[t]

    # Rumus pembersihan distribusi nilai mean p(x_{t-1} | x_t)
    mean = (1 / torch.sqrt(alpha_t)) * (x - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * noise_pred)

    if t > 0:
        noise = torch.randn_like(x)
        sigma_t = torch.sqrt(betas[t])
        return mean + sigma_t * noise
    return mean

@torch.no_grad()
def generate_samples(model, n_samples=2000):
    """Membangkitkan data baru dengan membalikkan arah noise dari koordinat acak."""
    x = torch.randn(n_samples, 2, device=DEVICE) # Memulai dari koordinat pure acak
    for t in reversed(range(T)):
        x = p_sample(model, x, t)
    return x.cpu().numpy()

print("\nSedang menguji proses sampling balik (Reverse Denoising)...")
samples = generate_samples(model, n_samples=2000)

# ------------------------------------------------------------------------------
# 8. VISUALIZATION & EXPORT
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Tampilan Pola Data Asli
axes[0].scatter(data[:, 0].cpu(), data[:, 1].cpu(), s=2, alpha=0.5)
axes[0].set_title("Original Data (Two Moons)")
axes[0].set_aspect('equal')

# Panel 2: Tampilan Data saat hancur oleh noise di tengah jalan (t = T/2)
xt_mid, _ = q_sample(data[:2000], torch.full((2000,), T // 2, device=DEVICE))
axes[1].scatter(xt_mid[:, 0].cpu(), xt_mid[:, 1].cpu(), s=2, alpha=0.5, color='orange')
axes[1].set_title(f"Noised Data (t = {T // 2})")
axes[1].set_aspect('equal')

# Panel 3: Tampilan Data baru hasil kreasi ulang model dari koordinat acak
axes[2].scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.5, color='green')
axes[2].set_title("Generated Samples (Denoised from Noise)")
axes[2].set_aspect('equal')

plt.tight_layout()
plt.savefig("diffusion_2d_toy.png", dpi=150)
print("\nSukses! Gambar grafik visualisasi disimpan dengan nama: 'diffusion_2d_toy.png'")
plt.show()

In [ ]:
from google.colab import files
import os

if os.path.exists('diffusion_2d_toy.png'):
    files.download('diffusion_2d_toy.png')
    print("Pop-up unduhan 'diffusion_2d_toy.png' berhasil dimunculkan!")
else:
    print("Berkas tidak ditemukan. Pastikan cell pengerjaan di atas dijalankan sampai tuntas.")

lab 4.5

In [ ]:
# ==============================================================================
# LAB 4.5 COMPLETE PIPELINE: FIXED FOR NEW LLAMA.CPP STANDARDS (FP16 -> Q4_0)
# ==============================================================================

import os
import torch
from google.colab import files

# Pastikan posisi awal eksekusi berada di direktori /content
%cd /content

print("=== [LANGKAH 1/5] Memasang Pustaka & Kloning llama.cpp ===")
!pip install transformers peft accelerator gguf --quiet
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp --quiet
print("Instalasi pustaka dan kloning llama.cpp sukses!\n")


print("=== [LANGKAH 2/5] Melakukan Kompilasi Pustaka llama.cpp ===")
%cd /content/llama.cpp
# Melakukan build binari quantize bawaan C++ agar siap digunakan
!make --quiet
print("Kompilasi selesai! Aplikasi utama siap digunakan.\n")


print("=== [LANGKAH 3/5] Proses Merging (Penggabungan) LoRA ke Base Model ===")
%cd /content
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

lora_folder = "./lora_finetuned_model"
merged_folder = "./merged_model_output"

print("Sedang menyatukan bobot adapter ke dalam base model...")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        lora_folder,
        torch_dtype=torch.float16,
        device_map="cpu"
    )
    model = PeftModel.from_pretrained(base_model, lora_folder)
    model = model.merge_and_unload()

    tokenizer = AutoTokenizer.from_pretrained(lora_folder)
    model.save_pretrained(merged_folder)
    tokenizer.save_pretrained(merged_folder)
    print(f"👉 SUKSES! Model utuh disimpan di: {merged_folder}\n")

    del base_model, model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"\n[⚠️ Peringatan saat Merge]: {e}")
    print("Menggunakan folder asli untuk langkah konversi langsung.")
    merged_folder = lora_folder


print("=== [LANGKAH 4/5] Konversi Valid ke Format Induk GGUF (FP16) ===")
!pip install -r llama.cpp/requirements.txt --quiet

print("Mengeksekusi skrip konversi resmi menggunakan parameter '--outtype f16'...")
# PERBAIKAN UTAMA: Menggunakan '--outtype f16' yang didukung penuh oleh aturan baru
if os.path.exists('llama.cpp/convert_hf_to_gguf.py'):
    !python3 llama.cpp/convert_hf_to_gguf.py {merged_folder} --outtype f16 --outfile model.gguf
else:
    !python3 llama.cpp/convert-hf-to-gguf.py {merged_folder} --outtype f16 --outfile model.gguf 2>/dev/null || \
     python3 llama.cpp/examples/convert_hf_to_gguf.py {merged_folder} --outtype f16 --outfile model.gguf


print("\n=== [LANGKAH 5/5] Pemotongan Ukuran ke 4-Bit (Kuantisasi Q4_0) & Download ===")
# Memberikan izin akses eksekusi pada aplikasi hasil kompilasi make
!chmod -R +x ./llama.cpp/

if os.path.exists('model.gguf'):
    print("Mengompresi bobot model dari 16-bit menjadi 4-bit (Tipe Kompatibel Q4_0)...")

    # Mengeksekusi pemotongan menggunakan binari hasil kompilasi langkah 2
    !./llama.cpp/llama-quantize model.gguf model_q4.gguf q4_0 2>/dev/null || \
     ./llama.cpp/quantize model.gguf model_q4.gguf q4_0 2>/dev/null

    # Menghapus file induk f16 agar RAM dan storage Colab tidak penuh
    if os.path.exists('model_q4.gguf'):
        !rm -rf model.gguf
        print("🔥 Luar Biasa! Berkas 'model_q4.gguf' sukses terbentuk secara sempurna.")
        print("Memicu sistem pop-up unduhan otomatis ke perangkat laptop Anda...")
        files.download('model_q4.gguf')
    else:
        print("❌ Gagal: File hasil kompresi 4-bit gagal diproduksi oleh aplikasi quantize.")
else:
    print("❌ Gagal: File induk 'model.gguf' tidak berhasil terbuat di Langkah 4.")

In [ ]:
# ==============================================================================
# JALUR PANDUAN PURE PYTHON COMPILER (PASTI JADI model_q4.gguf)
# ==============================================================================
import os
from google.colab import files

%cd /content

print("=== [LANGKAH 1/2] Sinkronisasi & Upgrade Modul Kuantisasi ===")
# Menginstal modul compiler 'gguf' dan menembakkan script eksekusi langsung
!pip install --upgrade gguf transformers huggingface_hub --quiet

print("\n=== [LANGKAH 2/2] Memaksa Pembuatan File model_q4.gguf ===")

# Menjalankan eksekutor Python murni tanpa perantara script llama.cpp yang usang
if os.path.exists('model.gguf'):
    print("Mengonversi file induk 'model.gguf' (988MB) ke format hemat 4-bit...")

    # Jalur Eksekusi A: Menggunakan modul resmi python gguf
    !python3 -m gguf.scripts.quantize model.gguf model_q4.gguf q4_0

    # Jalur Eksekusi B (Cadangan otomatis jika Jalur A diblokir oleh sistem)
    if not os.path.exists('model_q4.gguf'):
        print("Mencoba jalur compiler cadangan...")
        !python3 llama.cpp/examples/quantize/quantize.py model.gguf model_q4.gguf q4_0
else:
    print("Error: Berkas 'model.gguf' tidak ditemukan. Pastikan Anda melihatnya di panel kiri.")

# Validasi Akhir & Trigger Download Laptop
print("\n=== [VERIFIKASI AKHIR] ===")
if os.path.exists('model_q4.gguf'):
    print("🎉 SUKSES BESAR! File 'model_q4.gguf' akhirnya berhasil dibuat!")
    print("Menghapus berkas cadangan lama untuk menghemat penyimpanan...")
    !rm -rf model.gguf
    print("Memulai proses unduhan ke laptop Anda...")
    files.download('model_q4.gguf')
else:
    print("❌ File tetap menolak dibuat.")
    print("\n[SOLUSI DARURAT]: Karena file 'model.gguf' Anda yang berukuran 988MB sudah sukses,")
    print("kita akan langsung mengunduh file 988MB tersebut ke laptop Anda sekarang juga!")
    files.download('model.gguf')

In [ ]:
# ==============================================================================
# WORKFLOW POIN 3: FORMAL 4-BIT GGUF QUANTIZATION PIPELINE
# ==============================================================================

import os
from google.colab import files

# Pastikan berada di direktori /content
%cd /content

print("=== [PROSES] Memasang Compiler Kuantisasi Mandiri ===")
# Menginstal modul compiler resmi Hugging Face untuk pembuatan berkas GGUF
!pip install transformers peft accelerator gguf --quiet

print("\n=== [PROSES] Menjalankan Kuantisasi Langsung ke 4-Bit (Q4_0) ===")
# Kita gunakan modul internal python yang paling stabil untuk ekspor langsung ke q4_0
# Jalur ini membaca folder lora_finetuned_model milikmu yang sudah lengkap tadi

lora_folder = "./lora_finetuned_model"
output_file = "model_q4.gguf"

# Membersihkan sisa file corrupt lama jika ada
if os.path.exists(output_file):
    !rm -rf {output_file}

# Menjalankan skrip konversi Hugging Face ke GGUF dengan target kompresi 4-bit (q4_0)
if os.path.exists('llama.cpp/convert_hf_to_gguf.py'):
    !python3 llama.cpp/convert_hf_to_gguf.py {lora_folder} --outtype q4_0 --outfile {output_file}
else:
    # Antisipasi jika letak file bergeser di repositori terbaru
    !python3 llama.cpp/convert-hf-to-gguf.py {lora_folder} --outtype q4_0 --outfile {output_file} 2>/dev/null || \
     python3 llama.cpp/examples/convert_hf_to_gguf.py {lora_folder} --outtype q4_0 --outfile {output_file}

# ------------------------------------------------------------------------------
# VERIFIKASI AKHIR DAN PENGUNDUHAN
# ------------------------------------------------------------------------------
print("\n=== [VERIFIKASI AKHIR WORKFLOW POIN 3] ===")
if os.path.exists(output_file):
    print(f"🔥 SUKSES! Berkas '{output_file}' berukuran ekonomis 4-bit berhasil dibuat.")
    print("Memicu unduhan otomatis agar file bisa Anda masukkan ke proyek Flutter/React Native...")
    files.download(output_file)
else:
    print("❌ Format kuantisasi otomatis diblokir oleh lingkungan runtime.")
    print("Solusi Bypass Resmi: Mengompresi file 'model.gguf' yang sudah Anda download sebelumnya.")

    # Jika karena masalah memori script internal di atas mogok, kita gunakan taktik eksternal
    !python3 -m gguf.scripts.quantize model.gguf model_q4.gguf q4_0 2>/dev/null
    if os.path.exists(output_file):
        print(f"🔥 SUKSES VIA JALUR BACKPASS! File '{output_file}' siap diunduh.")
        files.download(output_file)

In [3]:
echo "# modul4" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/bintangF05/modul4.git
git push -u origin main

SyntaxError: invalid syntax (2088900017.py, line 1)